# ChatUniTest LoRA Fine-tuning

**Objective**: Fine-tune with CodeLlama-7b-Instruct + QLoRA to generate Java JUnit tests

**Estimated time**: 1.5-2 hours (A100)

**Estimated cost**: ~$2-3

**Pre-run checklist**:
- Runtime → Change runtime type → **A100 GPU**
- Ensure your HuggingFace token is ready (write permission required)

## Step 1: Install dependencies

In [ ]:
!pip install -q transformers==4.40.0 peft==0.10.0 trl==0.8.6 \
    bitsandbytes==0.43.1 accelerate==0.29.3 \
    datasets sentencepiece huggingface_hub

# Verify GPU
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Step 2: Log in to HuggingFace (write token required)

In [ ]:
from huggingface_hub import login, notebook_login

# Option 1: Interactive login (recommended)
notebook_login()

# Option 2: Paste token directly (not recommended in plain text)
# login(token="hf_xxxxxxxxxxxxx")

## Step 3: Prepare dataset

In [ ]:
from datasets import load_dataset
import pandas as pd
import re

# -- Prompt template (matches model_server.py inference format) --
PROMPT_TEMPLATE = """mode=COMPLETION
projectPath=unknown
assertionStyle=JUNIT
staticSnapshot:
{context}
runtimeFacts:

### JUnit Test:
"""

def build_full_text(context: str, test: str) -> str:
    return PROMPT_TEMPLATE.format(context=context.strip()) + test.strip()

def has_assertion(test: str) -> bool:
    return any(kw in test for kw in ["assert", "Assert", "verify", "Verify", "fail("])

def is_valid_java(code: str) -> bool:
    return code.count("{") > 0 and abs(code.count("{") - code.count("}")) <= 2

def estimate_tokens(text: str) -> int:
    return len(text) // 4

# Load dataset
print("Loading dataset...")
raw = load_dataset("zzzghttt/context2test", split="train")
print(f"Raw samples: {len(raw)}")
print(f"Columns: {raw.column_names}")

# Auto-detect column names
col_context = next((c for c in ["context", "input", "source"] if c in raw.column_names), None)
col_test    = next((t for t in ["test", "output", "target"] if t in raw.column_names), None)
print(f"Using: context='{col_context}', test='{col_test}')"

df = raw.to_pandas()[[col_context, col_test]].copy()
df.columns = ["context", "test"]

# -- Data cleaning --
before = len(df)
df = df.drop_duplicates(subset=["context"])
print(f"After deduplication: {len(df)} (removed {before - len(df)})")

df = df[df["test"].apply(has_assertion)]
print(f"After filtering samples without assertions: {len(df)}")

df = df[df["test"].apply(is_valid_java)]
print(f"After filtering structurally invalid samples: {len(df)}")

df = df[df["context"].str.strip().str.len() > 30]
print(f"After filtering empty contexts: {len(df)}")

# Build full training text
df["text"] = df.apply(lambda r: build_full_text(r["context"], r["test"]), axis=1)
df["token_est"] = df["text"].apply(estimate_tokens)
df = df[df["token_est"] <= 2048]
print(f"After filtering over-length samples: {len(df)}")

# Randomly keep 5000 samples
df = df.sample(frac=1, random_state=42).reset_index(drop=True).head(5000)
print(f"\nFinal training sample count: {len(df)}")

# Convert to HuggingFace Dataset
from datasets import Dataset
train_dataset = Dataset.from_pandas(df[["text"]])

# Preview the first sample
print("\n=== Sample preview (first 500 chars) ===")
print(train_dataset[0]["text"][:500])

## Step 4: Load base model (QLoRA 4-bit)

In [ ]:
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)

BASE_MODEL = "codellama/CodeLlama-7b-Instruct-hf"

# 4-bit QLoRA config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Loading model (4-bit)...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
model.config.use_cache = False
model.config.pretraining_tp = 1

print(f"Model loaded. VRAM allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## Step 5: Configure LoRA

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Prepare model for QLoRA training
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=32,                    # Original TestGen2-lora used 64; 32 is faster to train
    lora_alpha=64,           # alpha = 2*r for better stability
    target_modules=[
        "q_proj", "v_proj",  # Original config
        "k_proj", "o_proj",  # Extended: cover full attention
    ],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Expected output: trainable params: ~8M / 7B total (~0.1%)

## Step 6: Train

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

# -- Edit this: your HuggingFace username --
HF_USERNAME = "your-hf-username"   # <-- replace with yours
OUTPUT_MODEL = f"{HF_USERNAME}/my-testgen-lora"

training_args = TrainingArguments(
    output_dir="./my-testgen-lora",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=8,       # effective batch_size = 32
    gradient_checkpointing=True,         # save VRAM
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    fp16=False,
    bf16=True,                           # A100 supports bfloat16
    logging_steps=10,
    save_strategy="steps",
    save_steps=100,                      # Save checkpoint every 100 steps
    save_total_limit=3,
    report_to="none",                    # Do not upload to wandb
    optim="paged_adamw_32bit",           # QLoRA recommended optimizer
    max_grad_norm=0.3,
    group_by_length=True,                # Group similar lengths to reduce padding
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    args=training_args,
    tokenizer=tokenizer,
    dataset_text_field="text",
    max_seq_length=2048,                 # Match inference cutoff_len
    packing=False,
)

print("Starting training...")
print(f"Total steps: {len(train_dataset) // (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps)}")
trainer.train()

## Step 7: Save and push to HuggingFace Hub

In [ ]:
print("Saving model...")
trainer.save_model("./my-testgen-lora")

print(f"Pushing to HuggingFace Hub: {OUTPUT_MODEL}")
model.push_to_hub(OUTPUT_MODEL, private=False)
tokenizer.push_to_hub(OUTPUT_MODEL, private=False)

print(f"\nDone! Model URL: https://huggingface.co/{OUTPUT_MODEL}")
print(f"\nNext step: replace line 23 in model_server.py with:")
print(f'  PeftModel.from_pretrained(model, "{OUTPUT_MODEL}")')

## Step 8 (Optional): Resume training from checkpoint

If Colab disconnects, use the code below to resume from the latest checkpoint:

In [ ]:
import os

# Find the latest checkpoint
checkpoints = [
    d for d in os.listdir("./my-testgen-lora")
    if d.startswith("checkpoint-")
]
if checkpoints:
    latest = sorted(checkpoints, key=lambda x: int(x.split("-")[1]))[-1]
    resume_from = f"./my-testgen-lora/{latest}"
    print(f"Resuming training from {resume_from}")
    trainer.train(resume_from_checkpoint=resume_from)
else:
    print("No checkpoint found, start training from scratch")

## Step 9 (Optional): Quick generation sanity check

In [ ]:
from transformers import GenerationConfig

# Test with the same prompt format as model_server.py
test_prompt = """mode=COMPLETION
projectPath=unknown
assertionStyle=JUNIT
staticSnapshot:
public class PDFTextStripper {
    public String getText(PDDocument doc) throws IOException {
        StringWriter writer = new StringWriter();
        writeText(doc, writer);
        return writer.toString();
    }
}
runtimeFacts:

### JUnit Test:
"""

model.eval()
inputs = tokenizer(test_prompt, return_tensors="pt", truncation=True, max_length=2048).to("cuda")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.6,
        do_sample=True,
        top_p=0.95,
        repetition_penalty=1.1,
        eos_token_id=tokenizer.eos_token_id,
    )

generated = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print("=== Generated output ===")
print(generated)